In [ ]:
import streamlit as st
import pandas as pd
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_auc_score, matthews_corrcoef

# Page Config
st.set_page_config(page_title="Bank Marketing Predictor", layout="wide")

st.title("🏦 Bank Marketing Campaign Prediction")
st.markdown("""
This app predicts whether a customer will subscribe to a **Term Deposit** based on campaign data.
Choose a model from the sidebar and upload your test data to see the results.
""")

# --- SIDEBAR ---
st.sidebar.header("Model Settings")
model_choice = st.sidebar.selectbox(
    "Select ML Model", 
    ["Logistic_Regression", "Decision_Tree", "kNN", "Naive_Bayes", "Random_Forest", "XGBoost"]
)

# --- FILE UPLOADER ---
uploaded_file = st.file_uploader("Upload Test CSV (Ensure it is Semicolon ';' separated)", type="csv")

# Function to load models and helpers
def load_assets(model_name):
    with open(f'{model_name}.pkl', 'rb') as f:
        model = pickle.load(f)
    with open('scaler.pkl', 'rb') as f:
        scaler = pickle.load(f)
    with open('label_encoders.pkl', 'rb') as f:
        encoders = pickle.load(f)
    return model, scaler, encoders

if uploaded_file is not None:
    # Read the data
    test_df = pd.read_csv(uploaded_file, sep=';')
    st.subheader("📊 Uploaded Data Preview")
    st.dataframe(test_df.head())

    # Load assets
    model, scaler, encoders = load_assets(model_choice)

    # Preprocessing copy
    proc_df = test_df.copy()
    
    # 1. Apply Label Encoding to categorical columns
    for col, le in encoders.items():
        if col in proc_df.columns:
            # Map categories to numbers, handle unseen data with -1
            proc_df[col] = proc_df[col].map(lambda s: le.transform([s])[0] if s in le.classes_ else -1)

    # 2. Separate Features and Target
    if 'y' in proc_df.columns:
        X_test = proc_df.drop('y', axis=1)
        y_test = proc_df['y'].map({'no': 0, 'yes': 1}) # Convert target to binary
    else:
        X_test = proc_df
        y_test = None

if st.button(f"🚀 Predict using {model_choice}"):
        # Scale the data
        X_test_scaled = scaler.transform(X_test)
        
        # Predict
        predictions = model.predict(X_test_scaled)
        
        # Display Results
        st.success(f"Prediction Complete using {model_choice}!")
        
        # If ground truth 'y' exists, show metrics
        if y_test is not None:
            col1, col2, col3 = st.columns(3)
            col1.metric("Accuracy", f"{accuracy_score(y_test, predictions):.4f}")
            col2.metric("MCC", f"{matthews_corrcoef(y_test, predictions):.4f}")
            
            # Confusion Matrix Visualization
            st.subheader("📈 Confusion Matrix")
            fig, ax = plt.subplots()
            cm = confusion_matrix(y_test, predictions)
            sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=ax)
            ax.set_xlabel('Predicted')
            ax.set_ylabel('Actual')
            st.pyplot(fig)
            
            # Classification Report
            st.subheader("📝 Detailed Classification Report")
            st.text(classification_report(y_test, predictions))